# Mega Ensemble: LGBM + XGBoost + CatBoost & Spatial Features
Eksperimen mutakhir untuk mencapai skor RMSE optimal di Leaderboard Kaggle.

Strategi:
- Menambahkan **Spatial K-Means Clustering** untuk memisahkan pos pantau ke dalam beberapa zona geografis.
- Menggunakan **Optuna** untuk *hyperparameter tuning* otomatis.
- Menggabungkan kekuatan tiga raksasa Gradient Boosting: **LightGBM**, **XGBoost**, dan **CatBoost** dalam satu *blending ensemble*.
- Aturan bersih: Kode bebas komentar, logika dijelaskan di Markdown.

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)

## 1. Load Dataset & Spatial Clustering
Memuat data dari folder mentah, lalu melakukan *clustering* pada koordinat pos pantau menggunakan K-Means (membagi pos ke dalam 5 zona geografis).

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

kmeans = KMeans(n_clusters=5, random_state=42)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

train['datetime'] = pd.to_datetime(train['datetime'])
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]

## 2. Preprocessing & Agregasi Data Lingkungan
Merangkum data resolusi 1-jam menjadi 3-jam. Curah hujan dihitung akumulasinya (sum), sedangkan fitur suhu dan iklim diambil rata-ratanya (mean). Nilai kosong (missing values) diatasi dengan interpolasi forward dan backward fill.

In [3]:
def aggregate_env_data(df):
    df_sorted = df.sort_values(by=['nama_pos', 'datetime'])
    cat_cols = ['nama_pos', 'landcover_name', 'datetime']
    num_cols = [c for c in df.columns if c not in cat_cols]
    
    agg_funcs = {col: 'mean' for col in num_cols}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    
    df_indexed = df_sorted.set_index('datetime')
    agg_df = df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()
    return agg_df

env_agg = aggregate_env_data(env_data)
env_agg = env_agg.sort_values(['nama_pos', 'datetime'])
env_agg = env_agg.groupby('nama_pos', group_keys=False).apply(lambda group: group.ffill().bfill())

## 3. Merge & Feature Engineering
Penggabungan seluruh data menjadi satu *dataframe* utuh. Fitur tambahan dibuat berdasarkan perhitungan waktu silik (Cyclical Time dengan Sinus/Cosinus) dan pergerakan jendela akumulasi curah hujan (Rolling Sum 12 & 24 jam).

In [4]:
train_df = pd.merge(train, env_agg, on=['datetime', 'nama_pos'], how='left')
test_df = pd.merge(test, env_agg, on=['datetime', 'nama_pos'], how='left')

train_df = pd.merge(train_df, coords, on='nama_pos', how='left')
test_df = pd.merge(test_df, coords, on='nama_pos', how='left')

def engineer_features(df):
    df['month'] = df['datetime'].dt.month
    df['hour'] = df['datetime'].dt.hour
    
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
    
    df = df.sort_values(by=['nama_pos', 'datetime'])
    df['rainfall_rolling_12h_sum'] = df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=4, min_periods=1).sum())
    df['rainfall_rolling_24h_sum'] = df.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=8, min_periods=1).sum())
    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

le = LabelEncoder()
train_df['nama_pos_encoded'] = le.fit_transform(train_df['nama_pos'])
test_df['nama_pos_encoded'] = le.transform(test_df['nama_pos'])

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id', 'landcover_name']
features = [c for c in train_df.columns if c not in drop_cols]
target = 'tma_mdpl'

## 4. Local Validation Split
Data pelatihan diurutkan berdasarkan waktu. 70% data awal digunakan untuk pelatihan (*training*) dan 30% terakhir digunakan untuk pengujian (*validation*) guna mensimulasikan lingkungan *leaderboard* masa depan.

In [5]:
train_df = train_df.sort_values('datetime')
split_idx = int(len(train_df) * 0.7)

X_train, y_train = train_df.iloc[:split_idx][features], train_df.iloc[:split_idx][target]
X_val, y_val = train_df.iloc[split_idx:][features], train_df.iloc[split_idx:][target]

## 5. Hyperparameter Tuning (Optuna)
Menjalankan simulasi optimalisasi ringan menggunakan Optuna untuk 3 model (LightGBM, XGBoost, CatBoost).
Demi durasi komputasi yang efisien pada fase eksperimen kali ini, percobaan dibatasi maksimal 5 putaran pencarian (*trials*) per model.

In [6]:
def objective_lgb(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'random_state': 42,
        'verbose': -1
    }
    model = lgb.LGBMRegressor(**params, n_estimators=100)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 5, 9),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params, n_estimators=100)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

def objective_cat(trial):
    params = {
        'loss_function': 'RMSE',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 5, 8),
        'random_seed': 42,
        'verbose': False
    }
    model = CatBoostRegressor(**params, iterations=100)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

optuna.logging.set_verbosity(optuna.logging.WARNING)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=5)
best_lgb_params = study_lgb.best_params
best_lgb_params['objective'] = 'regression'
best_lgb_params['random_state'] = 42
best_lgb_params['verbose'] = -1

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=5)
best_xgb_params = study_xgb.best_params
best_xgb_params['objective'] = 'reg:squarederror'
best_xgb_params['random_state'] = 42

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=5)
best_cat_params = study_cat.best_params
best_cat_params['loss_function'] = 'RMSE'
best_cat_params['random_seed'] = 42
best_cat_params['verbose'] = False

## 6. Training Model Global
Melatih ketiga arsitektur algoritma (LightGBM, XGBoost, CatBoost) menggunakan parameter terbaik dari hasil temuan Optuna pada seluruh dataset (*Full Data*).

In [7]:
model_lgb = lgb.LGBMRegressor(**best_lgb_params, n_estimators=500)
model_lgb.fit(train_df[features], train_df[target])

model_xgb = xgb.XGBRegressor(**best_xgb_params, n_estimators=500)
model_xgb.fit(train_df[features], train_df[target])

model_cat = CatBoostRegressor(**best_cat_params, iterations=500)
model_cat.fit(train_df[features], train_df[target])

CatBoostRegressor(depth=8, iterations=500, learning_rate=0.060546395758931996, loss_function='RMSE', random_seed=42, verbose=False)

## 7. Mega Ensemble Blending & Submission
Melakukan penarikan prediksi dari 3 model tersebut ke data pengujian (*test*). Prediksi final merupakan rata-rata tertimbang (*weighted average*) dari ketiga model untuk meminimalisasi *overfitting* individu.
Hasil akhir dicetak ketat ke file `submissions/submission.csv` sesuai SOP.

In [8]:
preds_lgb = model_lgb.predict(test_df[features])
preds_xgb = model_xgb.predict(test_df[features])
preds_cat = model_cat.predict(test_df[features])

final_preds = (preds_lgb * 0.4) + (preds_xgb * 0.3) + (preds_cat * 0.3)

sub = pd.DataFrame({
    'id': test_df['id'],
    'tma_mdpl': final_preds
})

sub.to_csv('../submissions/submission.csv', index=False)
print("Mega Ensemble berhasil diselesaikan.")
print("File tersimpan: submissions/submission.csv")

Mega Ensemble berhasil diselesaikan.
File tersimpan: submissions/submission.csv
